In [ ]:
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# !pip install hf_transfer polars sentence_transformers numpy scipy transformers torch accelerate notebook ipywidgets jupyter_contrib_nbextensions
# !jupyter contrib nbextension install --user
# !jupyter nbextension enable --py widgetsnbextension

Traceback (most recent call last):
  File "/usr/local/bin/jupyter-contrib", line 8, in <module>
    sys.exit(main())
  File "/root/.local/lib/python3.8/site-packages/jupyter_core/application.py", line 283, in launch_instance
    super().launch_instance(argv=argv, **kwargs)
  File "/root/.local/lib/python3.8/site-packages/traitlets/config/application.py", line 1073, in launch_instance
    app = cls.instance(**kwargs)
  File "/root/.local/lib/python3.8/site-packages/traitlets/config/configurable.py", line 583, in instance
    inst = cls(*args, **kwargs)
  File "/usr/local/lib/python3.8/dist-packages/jupyter_contrib_core/application.py", line 27, in __init__
    self._refresh_subcommands()
  File "/usr/local/lib/python3.8/dist-packages/jupyter_contrib_core/application.py", line 43, in _refresh_subcommands
    get_subcommands_dict = entrypoint.load()
  File "/usr/lib/python3/dist-packages/pkg_resources/__init__.py", line 2444, in load
    self.require(*args, **kwargs)
  File "/usr/lib/pyth

In [2]:
from typing import List, Dict
import numpy as np
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

class RAGSystem:
    def __init__(self, documents: List[str], 
                 embedding_model_name: str,
                 llm_model_name: str):
        """
        Initialize the RAG system with documents and models.
        
        Args:
            documents: List of text documents for the knowledge base
            embedding_model_name: Model for creating embeddings
            llm_model_name: Qwen model to use for generation
        """
        self.documents = documents
        
        # Initialize embedding model
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.document_embeddings = self.embedding_model.encode(documents,  show_progress_bar=True)
        
        # Initialize Qwen model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            llm_model_name,
            device_map="auto",
            trust_remote_code=True
        ).eval()
    
    def retrieve(self, query: str, k: int = 3) -> List[str]:
        """
        Retrieve the k most relevant documents for a given query.
        
        Args:
            query: The user's question or query
            k: Number of documents to retrieve
            
        Returns:
            List of the k most relevant documents
        """
        # Get query embedding
        query_embedding = self.embedding_model.encode([query])[0]
        
        # Calculate similarities
        similarities = [
            1 - cosine(query_embedding, doc_embedding)
            for doc_embedding in self.document_embeddings
        ]
        
        # Get top k documents
        top_k_indices = np.argsort(similarities)[-k:][::-1]
        
        return [self.documents[i] for i in top_k_indices]
    
    def generate_response(self, query: str, retrieved_docs: List[str]) -> str:
        """
        Generate a response using Qwen based on the query and retrieved documents.
        
        Args:
            query: The user's question
            retrieved_docs: List of relevant documents to use as context
            
        Returns:
            Generated response
        """
        # Construct the prompt
        context = "\n".join(retrieved_docs)
        prompt = f"""Context information is below.
        ---------------------
        {context}
        ---------------------
        Given the context information above, please answer the query: {query}
        Answer:"""
        
        # Format prompt for Qwen
        messages = [
            {"role": "system", "content": "You are a helpful assistant that answers questions based on the provided context."},
            {"role": "user", "content": prompt}
        ]
        
        # Convert messages to Qwen chat format
        query_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Generate response
        inputs = self.tokenizer(query_text, return_tensors='pt').to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.7,
                top_p=0.9,
                do_sample=True
            )
        
        response = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return response.strip()
    
    def query(self, query: str, k: int = 3) -> Dict:
        """
        Complete RAG pipeline: retrieve documents and generate response.
        
        Args:
            query: The user's question
            k: Number of documents to retrieve
            
        Returns:
            Dictionary containing the response and retrieved documents
        """
        retrieved_docs = self.retrieve(query, k)
        response = self.generate_response(query, retrieved_docs)
        
        return {
            "response": response,
            "retrieved_documents": retrieved_docs
        }

/usr/local/lib/python3.8/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [5]:
import polars as pl

read_df = pl.read_parquet('../data/driversf.parquet')

read_df.shape

(250, 14)

In [37]:
!pip install tqdm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [1]:
texts = read_df['text'].to_list()

# Initialize RAG system
rag = RAGSystem(
    documents=texts,
    embedding_model_name="sentence-transformers/all-mpnet-base-v2",
    llm_model_name="Qwen/Qwen2.5-0.5B-Instruct"
)

query = "What specific mechanics make this game unique?"

result = rag.query(query)
print(f"Query: {query}")
print(f"\nResponse: {result['response']}")

NameError: name 'read_df' is not defined